<a href="https://colab.research.google.com/github/Miranita-ar/Skripsi-Gojek-App-Review/blob/main/Code/((B)(No_CW))_Skripsi_IndoBERT_(Split_Data-Tokenisasi).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install & Import Library

In [ ]:
# =======================================================
# SEL 1 : INSTALL LIBRARY
# =======================================================

!pip install -q transformers datasets accelerate evaluate
!pip install -q scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
# =======================================================
# SEL 1a : IMPORT LIBRARY
# =======================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import AutoTokenizer

pd.set_option("display.max_columns", None)

# Mount Google Drive

In [ ]:
# =======================================================
# SEL 2 : MOUNT DRIVE & FOLDER
# =======================================================

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight"


!mkdir -p "{PROJECT_DIR}/data/split"
!mkdir -p "{PROJECT_DIR}/data/tokenized"
!mkdir -p "{PROJECT_DIR}/data/tokenizer"

!mkdir -p "{PROJECT_DIR}/analysis"
!mkdir -p "{PROJECT_DIR}/models"
!mkdir -p "{PROJECT_DIR}/results"
!mkdir -p "{PROJECT_DIR}/logs"

print("✅ Struktur folder No Class Weight berhasil dibuat")

Mounted at /content/drive
✅ Struktur folder No Class Weight berhasil dibuat


# Load Data Preproceseesd (Notebook A)

In [ ]:
# =======================================================
# SEL 4 : LOAD DATA PREPROCESSED
# =======================================================

DATA_URL = "https://raw.githubusercontent.com/Miranita-ar/Skripsi-Gojek-App-Review/refs/heads/main/Data/data_preprocessed_multiclass.csv"

df_raw = pd.read_csv(DATA_URL)

print("="*70)
print("DATA PREPROCESSED")
print("="*70)

print(f"\nJumlah Data : {len(df_raw):,}")

display(df_raw.head())

DATA PREPROCESSED

Jumlah Data : 19,794


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,label,content_lower_case,content_clean,content_normalisasi
0,5adef64a-687b-44cb-ae83-fccac29af5f8,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"saya sudah memakai aplikasi gojek 4 tahun,alha...",5,0,5.47.1,2026-02-03 00:57:36,NaN,NaN,5.47.1,2,"saya sudah memakai aplikasi gojek 4 tahun,alha...",saya sudah memakai aplikasi gojek 4 tahun alha...,saya sudah memakai aplikasi gojek 4 tahun alha...
1,ee4c68a5-56ba-4460-a302-8f6c1fe3b919,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,jarang ada diskon,5,0,5.48.2,2026-02-03 00:51:32,NaN,NaN,5.48.2,2,jarang ada diskon,jarang ada diskon,jarang ada diskon
2,a1487729-8a06-4fb9-a99e-762150ba7438,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"makin berat aja apk, muter mulu mau beli tiket...",1,0,5.39.1,2026-02-03 00:37:37,"Hai Kak @Now You See Me Sony Handoko, mohon ma...",2026-02-03 01:46:54,5.39.1,0,"makin berat aja apk, muter mulu mau beli tiket...",makin berat aja apk muter mulu mau beli tiket krl,makin berat aja aplikasi muter terus mau beli ...
3,37bb5ff1-22d2-49aa-afa9-4c0a88564a8e,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aplikasi terbaik,5,0,5.47.1,2026-02-03 00:27:07,NaN,NaN,5.47.1,2,aplikasi terbaik,aplikasi terbaik,aplikasi terbaik
4,06afd47c-c6be-4839-9931-9d90b918b039,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Di kasih limit di gopay pinjam 200rb,pengajuan...",2,0,5.48.2,2026-02-03 00:24:42,"Hai Kak Wirijaf, mohon maaf atas ketidaknyaman...",2026-02-03 01:42:50,5.48.2,0,"di kasih limit di gopay pinjam 200rb,pengajuan...",di kasih limit di gopay pinjam 200rb pengajuan...,di kasih limit di gopay pinjam 200 ribu pengaj...


In [ ]:
# =======================================================
# SEL 5 : DISTRIBUSI LABEL AWAL
# =======================================================


df = df_raw[["content_normalisasi", "label"]].copy()

df = df.rename(
    columns={
        "content_normalisasi":"text"
    }
)

df["label"] = df["label"].astype(int)

df = df.dropna(subset=["text"])

df = df[
    df["text"].str.strip() != ""
]


label_dist = (
    df["label"]
    .value_counts()
    .sort_index()
)

label_percent = (
    df["label"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

summary_awal = pd.DataFrame({
    "Jumlah": label_dist,
    "Persentase (%)": label_percent.round(2)
})

print("=== DISTRIBUSI LABEL AWAL ===")

display(summary_awal)

=== DISTRIBUSI LABEL AWAL ===


,Jumlah,Persentase (%)
label,,
0,6122,30.93
1,757,3.82
2,12915,65.25


In [ ]:
# =======================================================
# SEL 5a : PERSIAPAN DATA TRAINING
# =======================================================

df = df_raw[["content_normalisasi", "label"]].copy()

df = df.rename(
    columns={
        "content_normalisasi":"text"
    }
)

df["label"] = df["label"].astype(int)

df = df.dropna(subset=["text"])

df = df[
    df["text"].str.strip() != ""
]

print(f"Jumlah data siap training : {len(df):,}")

display(df.head())

Jumlah data siap training : 19,794


,text,label
0,saya sudah memakai aplikasi gojek 4 tahun alha...,2
1,jarang ada diskon,2
2,makin berat aja aplikasi muter terus mau beli ...,0
3,aplikasi terbaik,2
4,di kasih limit di gopay pinjam 200 ribu pengaj...,0


# Pembagian Data

In [ ]:
# =======================================================
# SEL 6 : STRATIFIED SPLIT
# =======================================================

SEED = 42

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

print("="*70)
print("HASIL SPLIT")
print("="*70)

print(f"Train : {len(train_df):,}")
print(f"Valid : {len(valid_df):,}")
print(f"Test  : {len(test_df):,}")

HASIL SPLIT
Train : 15,835
Valid : 1,979
Test  : 1,980


In [ ]:
# =======================================================
# SEL 7 : DISTRIBUSI SPLIT
# =======================================================

def distribusi(df_input, nama):

    temp = pd.DataFrame()

    temp["Jumlah"] = (
        df_input["label"]
        .value_counts()
        .sort_index()
    )

    temp["Persentase"] = (
        df_input["label"]
        .value_counts(normalize=True)
        .sort_index()*100
    ).round(2)

    print(f"\n=== {nama} ===")

    display(temp)

    return temp

train_dist = distribusi(train_df, "TRAIN")
valid_dist = distribusi(valid_df, "VALID")
test_dist = distribusi(test_df, "TEST")


=== TRAIN ===


,Jumlah,Persentase
label,,
0,4897,30.93
1,606,3.83
2,10332,65.25



=== VALID ===


,Jumlah,Persentase
label,,
0,612,30.92
1,76,3.84
2,1291,65.23



=== TEST ===


,Jumlah,Persentase
label,,
0,613,30.96
1,75,3.79
2,1292,65.25


In [ ]:
# =======================================================
# SEL 8 : SIMPAN CSV SPLIT
# =======================================================

train_df.to_csv(
    f"{PROJECT_DIR}/data/split/train_multiclass_noCW.csv",
    index=False
)

valid_df.to_csv(
    f"{PROJECT_DIR}/data/split/valid_multiclass_noCW.csv",
    index=False
)

test_df.to_csv(
    f"{PROJECT_DIR}/data/split/test_multiclass_noCW.csv",
    index=False
)

print("✅ CSV split berhasil disimpan")

✅ CSV split berhasil disimpan


In [ ]:
# title
# =======================================================
# SEL 8 : SIMPAN DATA SPLIT
# =======================================================

train_df.to_csv(
    f"{PROJECT_DIR}/train_multiclass_noCW.csv",
    index=False
)

valid_df.to_csv(
    f"{PROJECT_DIR}/valid_multiclass_noCW.csv",
    index=False
)

test_df.to_csv(
    f"{PROJECT_DIR}/test_multiclass_noCW.csv",
    index=False
)

print("train_multiclass_noCW.csv tersimpan")
print("valid_multiclass_noCW.csv tersimpan")
print("test_multiclass_noCW.csv tersimpan")

train_multiclass_noCW.csv tersimpan
valid_multiclass_noCW.csv tersimpan
test_multiclass_noCW.csv tersimpan


# Hitung Class Weight

karena tanpa class weight jadinya tidak pakai class weight gunakan train set saja

# Load Tokenizer IndoBERT

In [ ]:
# =======================================================
# SEL 11 : LOAD TOKENIZER
# =======================================================

tokenizer = AutoTokenizer.from_pretrained(
    "indobenchmark/indobert-base-p2"
)

print("✅ Tokenizer berhasil dimuat")

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Tokenizer berhasil dimuat


In [ ]:
# =======================================================
# SEL 12 : TOKENISASI
# =======================================================

MAX_LENGTH = 128

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

valid_dataset = valid_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

print("✅ Tokenisasi selesai")

Map:   0%|          | 0/15835 [00:00<?, ? examples/s]

Map:   0%|          | 0/1979 [00:00<?, ? examples/s]

Map:   0%|          | 0/1980 [00:00<?, ? examples/s]

✅ Tokenisasi selesai


In [ ]:
# =======================================================
# SEL 13 : SIMPAN TOKENIZED DATASET
# =======================================================

train_dataset.save_to_disk(
    f"{PROJECT_DIR}/data/tokenized/train_dataset_noCW"
)

valid_dataset.save_to_disk(
    f"{PROJECT_DIR}/data/tokenized/valid_dataset_noCW"
)

test_dataset.save_to_disk(
    f"{PROJECT_DIR}/data/tokenized/test_dataset_noCW"
)

print("✅ Dataset tokenized tersimpan")

Saving the dataset (0/1 shards):   0%|          | 0/15835 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1979 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1980 [00:00<?, ? examples/s]

✅ Dataset tokenized tersimpan


In [ ]:
# =======================================================
# SEL 14 : SIMPAN TOKENIZER
# =======================================================

tokenizer.save_pretrained(
    f"{PROJECT_DIR}/data/tokenizer"
)

print("✅ Tokenizer tersimpan")

✅ Tokenizer tersimpan


# Simpan Hasil

In [ ]:
# =======================================================
# SEL 15 : SPLIT SUMMARY
# =======================================================

split_summary = pd.concat(
    {
        "train": train_dist["Jumlah"],
        "valid": valid_dist["Jumlah"],
        "test": test_dist["Jumlah"]
    },
    axis=1
)

split_summary.to_csv(
    f"{PROJECT_DIR}/analysis/split_summary_noCW.csv"
)

display(split_summary)

,train,valid,test
label,,,
0,4897,612,613
1,606,76,75
2,10332,1291,1292


In [ ]:
# =======================================================
# SEL 16 : DISTRIBUSI PERSENTASE
# =======================================================

split_distribution = pd.concat(
    {
        "train": train_dist["Persentase"],
        "valid": valid_dist["Persentase"],
        "test": test_dist["Persentase"]
    },
    axis=1
)

split_distribution.to_csv(
    f"{PROJECT_DIR}/analysis/split_distribution_noCW.csv"
)

display(split_distribution)

,train,valid,test
label,,,
0,30.93,30.92,30.96
1,3.83,3.84,3.79
2,65.25,65.23,65.25


In [ ]:
# =======================================================
# SEL 17 : VERIFIKASI FILE
# =======================================================

for root, dirs, files in os.walk(PROJECT_DIR):

    level = root.replace(PROJECT_DIR, '').count(os.sep)

    indent = ' ' * 4 * level

    print(f'{indent}{os.path.basename(root)}/')

    subindent = ' ' * 4 * (level + 1)

    for f in files:
        print(f'{subindent}{f}')

No_Class_Weight/
    data/
        split/
            train_multiclass_noCW.csv
            valid_multiclass_noCW.csv
            test_multiclass_noCW.csv
        tokenized/
            train_dataset_noCW/
                data-00000-of-00001.arrow
                state.json
                dataset_info.json
            valid_dataset_noCW/
                data-00000-of-00001.arrow
                state.json
                dataset_info.json
            test_dataset_noCW/
                data-00000-of-00001.arrow
                state.json
                dataset_info.json
        tokenizer/
            tokenizer_config.json
            tokenizer.json
    analysis/
        split_summary_noCW.csv
        split_distribution_noCW.csv
    models/
    results/
    logs/


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/Skripsi_IndoBERT"):
    if "train_multiclass.csv" in files or "train_multiclass_noCW.csv" in files:
        print(root)
        print(files)
        print("="*50)

/content/drive/MyDrive/Skripsi_IndoBERT/data/split
['valid_multiclass.csv', 'train_multiclass.csv', 'test_multiclass.csv']
/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight
['train_multiclass_noCW.csv', 'valid_multiclass_noCW.csv', 'test_multiclass_noCW.csv']
/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight/data/split
['train_multiclass.csv', 'valid_multiclass.csv', 'test_multiclass.csv', 'train_multiclass_noCW.csv', 'valid_multiclass_noCW.csv', 'test_multiclass_noCW.csv']


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['CamScanner 06-26-2022 21.54_2.jpg', 'Tanda Tangan_1 (2).jpg', 'Tanda Tangan_1 (1).jpg', 'Tanda Tangan_1.jpg', 'SEMESTER 4', 'Kuliah', 'Screenshot_20240519-211708.jpg', 'Colab Notebooks', 'Dokumen tanpa judul (5).gdoc', 'Untitled spreadsheet (12).gsheet', 'KUIS_4B_ALGORITMA&STRUKTURDATA_2024_11220940000055_MIRANITA ANISA ROHMAH_4B.gdoc', 'Dokumen tanpa judul (4).gdoc', 'Belajar Nulis di Google Document.gdoc', 'Magang PT. Binar Edukasi Bangsa ', 'Lomba 17-an 2024', 'Untitled spreadsheet (11).gsheet', '[PEMBAHASAN-PU] Paket Soal 015.gdoc', 'Semester 5', 'Classroom', 'Dokumen tanpa judul (3).gdoc', 'ppt 8-9.mp4', 'swp55_crack.rar', 'Kenangan Kuliah', 'KULIAH', 'Adk3.gdoc', 'Stopword.gsheet', '5. Contoh 1.mp4', 'T7_055_MIRANITA ANISA ROHMAH_BASIS DATA.pdf', 'UAS SISTEM BASIS DATA', 'UTS-PDP_11220940000055_Miranita Anisa Rohmah.pdf', 'UAS-PDP_11220940000055_Miranita Anisa Rohmah.pdf', 'EXCEL', 'Salinan dari Menemukan dan Menghilangkan Duplikat.xlsx', 'dalam dunia saham apa itu lot dan apa 

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "No_Class_Weight" in dirs:
        print("Ditemukan di:")
        print(root)

Ditemukan di:
/content/drive/MyDrive/Skripsi_IndoBERT


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if "train_multiclass" in file:
            print(os.path.join(root, file))

/content/drive/MyDrive/Skripsi_IndoBERT/data/split/train_multiclass.csv
/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight/train_multiclass_noCW.csv
/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight/data/split/train_multiclass.csv
/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight/data/split/train_multiclass_noCW.csv


In [ ]:
import os

folder_split = "/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight/data/split"

files_to_delete = [
    "train_multiclass.csv",
    "valid_multiclass.csv",
    "test_multiclass.csv"
]

for file in files_to_delete:
    file_path = os.path.join(folder_split, file)

    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"✅ Dihapus: {file}")
    else:
        print(f"⚠️ Tidak ditemukan: {file}")

✅ Dihapus: train_multiclass.csv
✅ Dihapus: valid_multiclass.csv
✅ Dihapus: test_multiclass.csv


In [ ]:
import shutil
import os

folder_tokenized = "/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight/data/tokenized"

folders_to_delete = [
    "train_dataset",
    "valid_dataset",
    "test_dataset"
]

for folder in folders_to_delete:
    folder_path = os.path.join(folder_tokenized, folder)

    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"✅ Dihapus: {folder}")
    else:
        print(f"⚠️ Tidak ditemukan: {folder}")

✅ Dihapus: train_dataset
✅ Dihapus: valid_dataset
✅ Dihapus: test_dataset


In [ ]:
import shutil
import os

folder_path = "/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight/data/tokenizer"

if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
    print("✅ Folder tokenizer dihapus")
else:
    print("⚠️ Folder tokenizer tidak ditemukan")

In [ ]:
import os

root_folder = "/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight"

files_to_delete = [
    "train_multiclass_noCW.csv",
    "valid_multiclass_noCW.csv",
    "test_multiclass_noCW.csv"
]

for file in files_to_delete:
    file_path = os.path.join(root_folder, file)

    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"✅ Dihapus: {file}")
    else:
        print(f"⚠️ Tidak ditemukan: {file}")

✅ Dihapus: train_multiclass_noCW.csv
✅ Dihapus: valid_multiclass_noCW.csv
✅ Dihapus: test_multiclass_noCW.csv


In [ ]:
# =======================================================
# SEL 2 : MOUNT DRIVE & FOLDER
# =======================================================

from google.colab import drive
drive.mount('/content/drive')

# Folder utama penelitian
ROOT_DIR = "/content/drive/MyDrive/Skripsi_IndoBERT"

# Folder khusus eksperimen No Class Weight
PROJECT_DIR = f"{ROOT_DIR}/No_Class_Weight"

# =======================================================
# Membuat struktur folder
# =======================================================

folders = [
    f"{PROJECT_DIR}/data/split",
    f"{PROJECT_DIR}/data/tokenized",
    f"{PROJECT_DIR}/data/tokenizer",
    f"{PROJECT_DIR}/analysis",
    f"{PROJECT_DIR}/models",
    f"{PROJECT_DIR}/results",
    f"{PROJECT_DIR}/logs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("="*60)
print("✅ Struktur folder No Class Weight berhasil dibuat")
print("="*60)
print(PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Struktur folder No Class Weight berhasil dibuat
/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight


In [ ]:
import shutil
import os

folder_path = "/content/drive/MyDrive/Skripsi_IndoBERT/No_Class_Weight/data/tokenizer_noCW"

if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
    print("✅ Folder tokenizer_noCW berhasil dihapus.")
else:
    print("⚠️ Folder tokenizer_noCW tidak ditemukan.")

✅ Folder tokenizer_noCW berhasil dihapus.
